# Stage 2: Data Preprocessing & Feature Engineering

This notebook demonstrates the preprocessing pipeline, which includes:
1. Handling missing values in `TotalCharges`.
2. Dropping redundant features like `customerID`.
3. Encoding categorical variables (Label Encoding for binary columns, One-Hot Encoding for multi-class columns).
4. Feature Engineering (`tenure_group`, `charges_per_month`).
5. Performing a stratified train-test split.
6. Feature Scaling of numerical attributes using `StandardScaler`.
7. Addressing target class imbalance using **SMOTE** and **SMOTEENN** techniques.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

# Set style
sns.set_theme(style="whitegrid")

## 1. Load Raw Dataset

In [ ]:
df = pd.read_csv('../data/raw/telco_churn.csv')
print(f"Raw dataset shape: {df.shape}")
df.head(2)

## 2. Missing Value Resolution & Cleaning
We will convert the whitespace strings in `TotalCharges` to `NaN`, parse them as numerical types, and impute them using the median value.

In [ ]:
# Cast spaces to NaN and float
df['TotalCharges'] = df['TotalCharges'].replace(r'^\s*$', np.nan, regex=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing value with median
tc_median = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(tc_median)

# Drop customerID column
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])
elif 'CustomerID' in df.columns:
    df = df.drop(columns=['CustomerID'])

print("Remaining nulls:", df.isnull().sum().sum())

## 3. Feature Engineering
We construct two features:
- `tenure_group`: Group tenure into 5 bins (0-12, 13-24, 25-48, 49-60, 61-72 months).
- `charges_per_month`: Computed as `TotalCharges` / `tenure`. If tenure is 0, we substitute `MonthlyCharges`.

In [ ]:
# Bins & labels for tenure
bins = [-1, 12, 24, 48, 60, 72]
labels = ['0-12', '13-24', '25-48', '49-60', '61-72']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels)

# Ratio charges_per_month
df['charges_per_month'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

print("Engineered Features Summary:")
print(df[['tenure', 'tenure_group', 'MonthlyCharges', 'TotalCharges', 'charges_per_month']].head(3))

## 4. Encoding Categorical Variables
Categorical columns with two categories (e.g. Yes/No, Male/Female) are mapped directly to binary outputs `0/1`. Multi-class features are encoded using `pd.get_dummies`.

In [ ]:
# Mapping binary features
binary_cols = {
    'gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1},
    'Churn': {'No': 0, 'Yes': 1}
}

for col, mapping in binary_cols.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

# One-Hot encode multi-class variables
categorical_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaymentMethod', 'tenure_group'
]

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Convert boolean dummies to integer
for col in df_encoded.columns:
    if df_encoded[col].dtype == bool:
        df_encoded[col] = df_encoded[col].astype(int)

print(f"Encoded dataset shape: {df_encoded.shape}")

## 5. Train-Test Split (80/20 Stratified)

In [ ]:
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## 6. Numerical Feature Scaling
To avoid data leakage, we fit the scaler exclusively on training data, then scale numerical features in both training and test sets.

In [ ]:
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_month']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

# Save scaler and feature columns order
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.joblib')
with open('../models/feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f)

print("Fitted scaler successfully saved.")

## 7. Handle Class Imbalance using SMOTE & SMOTEENN
We address our minority churn class undersampling with two techniques:
1. **SMOTE** (Synthetic Minority Over-sampling Technique)
2. **SMOTEENN** (SMOTE + Edited Nearest Neighbors)

In [ ]:
# Initial distribution
print("Original Train target distribution:", y_train.value_counts().to_dict())

# SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print("SMOTE Train target distribution:", y_train_smote.value_counts().to_dict())

# SMOTEENN
smoteenn = SMOTEENN(random_state=42)
X_train_smoteenn, y_train_smoteenn = smoteenn.fit_resample(X_train_scaled, y_train)
print("SMOTEENN Train target distribution:", y_train_smoteenn.value_counts().to_dict())

## 8. Export Processed Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

# Save test set
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save original train set
X_train_scaled.to_csv('../data/processed/X_train_original.csv', index=False)
y_train.to_csv('../data/processed/y_train_original.csv', index=False)

# Save SMOTE train set
X_train_smote.to_csv('../data/processed/X_train_smote.csv', index=False)
y_train_smote.to_csv('../data/processed/y_train_smote.csv', index=False)

# Save SMOTEENN train set
X_train_smoteenn.to_csv('../data/processed/X_train_smoteenn.csv', index=False)
y_train_smoteenn.to_csv('../data/processed/y_train_smoteenn.csv', index=False)

print("All processed datasets successfully saved to data/processed/")